In [3]:
%%capture
%pip install bitsandbytes==0.49.2 chromadb==1.5.9 peft==0.19.1 polars==1.42.1 \
    sentence-transformers==5.6.0 tokenizers==0.22.2 transformers==5.12.1

In [4]:
import os, gc, sys, shutil, subprocess, chromadb, logging

import numpy as np
import polars as pl

from pathlib import Path

from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from sentence_transformers import SentenceTransformer, CrossEncoder

from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM

from peft import get_peft_model, LoraConfig, TaskType
from peft.utils import set_peft_model_state_dict

from kaggle_secrets import UserSecretsClient

from transformers import logging as hf_logging
from transformers.utils.logging import disable_progress_bar

In [5]:
# Define data paths and verify chromadb data integrity
TEST_PATH = Path("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

CHROMA_DB_DATA = Path("/kaggle/input/datasets/spandanjit2005/knowledge-db")
CHROMA_DB_PATH = Path("knowledge_db")
CHROMA_DB_NAME = "knowledge_db"

if CHROMA_DB_PATH.exists():
    shutil.rmtree(CHROMA_DB_PATH)

shutil.copytree(CHROMA_DB_DATA, CHROMA_DB_PATH)

print("Copied files:\n")
for root, _, files in os.walk(CHROMA_DB_PATH):
    for f in files:
        p = os.path.join(root, f)
        print(f"{p}  ({os.path.getsize(p)} bytes)")

_verify_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
_verify_collection = _verify_client.get_collection(name=CHROMA_DB_NAME)
_count = _verify_collection.count()
assert _count > 0, "Collection is empty after copying from the Kaggle dataset"

_check = _verify_collection.get(limit=5, include=["embeddings"])
assert len(_check["ids"]) == 5, "Could not retrieve embeddings - vector segment may not have been copied"

_emb = np.array(_check["embeddings"][0], dtype=np.float32) # type: ignore
_sanity = _verify_collection.query(query_embeddings=[_emb.tolist()], n_results=1)
assert _sanity["ids"][0][0] == _check["ids"][0], \
    "Query did not return the vector's own nearest neighbor - index looks corrupted"

print(f"\nChroma DB copy verified: {_count} vectors present and searchable.")

del _verify_client, _verify_collection, _check, _emb, _sanity
gc.collect()

Copied files:

knowledge_db/chroma.sqlite3  (153993216 bytes)
knowledge_db/eadbd761-61c0-4d3b-bff6-313088f581d6/link_lists.bin  (105892 bytes)
knowledge_db/eadbd761-61c0-4d3b-bff6-313088f581d6/index_metadata.pickle  (712098 bytes)
knowledge_db/eadbd761-61c0-4d3b-bff6-313088f581d6/length.bin  (48952 bytes)
knowledge_db/eadbd761-61c0-4d3b-bff6-313088f581d6/header.bin  (100 bytes)
knowledge_db/eadbd761-61c0-4d3b-bff6-313088f581d6/data_level0.bin  (131729832 bytes)

Chroma DB copy verified: 12238 vectors present and searchable.


476

In [6]:
# Configure HuggingFace API Key
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_READ_TOKEN") if UserSecretsClient().get_secret("HF_READ_TOKEN") else "" # type: ignore

# Configure logging levels to hide model-loading report
logging.getLogger("transformers").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)

hf_logging.set_verbosity_error()

disable_progress_bar()

In [7]:
# Define quantization config, models, and intermediate file paths
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False
)

model_config = {
    "embedder_model": "Qwen/Qwen3-Embedding-4B",
    "reranker_model": "Qwen/Qwen3-Reranker-8B",
    "generative_slm": "Qwen/Qwen2.5-14B-Instruct"
}

LSTM_MODEL_PATH = "/kaggle/input/models/spandanjit2005/lstm-cross-attention-classifier/pytorch/default/2/lstm_model"
sys.path.insert(0, LSTM_MODEL_PATH)

BASE_RERANKER_PATH = "/kaggle/input/models/spandanjit2005/zerank-2-reranker-base/other/1/1"
MCQ_ADAPTER_PATH = "/kaggle/input/models/spandanjit2005/zerank-2-reranker-finetune/other/default/1"

inter_path_config = {
    "working_dir": Path.cwd(),
    
    "train_retrieved": Path("train_retrieved.parquet"),
    "test_retrieved": Path("test_retrieved.parquet"),
    "train_ranked": Path("train_ranked.parquet"),
    "test_ranked": Path("test_ranked.parquet"),
    "train_final": Path("train_final.parquet"),
    "test_final": Path("test_final.parquet"),

    "submission": Path("submission.csv")
}

In [8]:
# Define seeds, option cols, and model instructions
SEED = 42
OPTION_COLS = ["A", "B", "C", "D", "E"]

def reset_gpu() -> None:
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

MCQ_EMBED_INSTRUCTION = (
    "Given a multiple-choice science question and its answer options, retrieve document "
    "chunks that contain the information needed to determine the correct answer. "
    "Prioritize document chunks that have `Prompt:` and `Answer:` labels in them."
)

RAG_RERANK_INSTRUCTION = (
    "Given a multiple-choice question and its answer options, judge whether this "
    "chunk contains information that helps determine the correct answer. "
    "Rank document chunks that have `Prompt:` and `Answer:` labels in them higher."
)

GENERATIVE_SLM_INSTRUCTION = (
    "You are an expert scientist. Use the provided context to answer the multiple-choice "
    "question. If the context is irrelevant or conflicts with what you are confident is "
    "correct, rely on your own knowledge instead. "
    "Respond with only the letter of the single best option. Do not include any other text."
)

MCQ_RERANK_INSTRUCTION = (
    "Given a multiple-choice question, determine whether the candidate answer is the "
    "single factually correct answer to that question. Score it high only if it is "
    "precise, accurate, and directly resolves what the question is asking. Score it "
    "low if it is a plausible-sounding distractor, a partially correct answer, an "
    "answer that is topically related but does not actually answer the question, or "
    "one that is factually wrong. The questions span a wide range of domains "
    "physics, chemistry, biology, astronomy, mathematics, philosophy, and other "
    "STEM and humanities topics - so judge correctness using domain-appropriate "
    "reasoning, not surface-level keyword overlap between the question and the answer."
)

In [9]:
# Read test data, construct `mcq_query` and initialize chromadb client
test_data = pl.read_csv(TEST_PATH)

test_data = test_data.with_columns(
    (
        pl.lit("Prompt : ") + pl.col("prompt")
        + pl.lit(" Options: ")
        + pl.concat_str(
            [pl.lit(f"{opt}) ") + pl.col(opt).fill_null(" ") for opt in OPTION_COLS],
            separator=" "
        )
    ).alias("mcq_query")
)

chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH) 
collection = chroma_client.get_collection(name=CHROMA_DB_NAME)

In [10]:
# Define function to retrieve top `k` chunks for every query [retrieve_top_k]
def retrieve_top_k(
    query_embeddings: np.ndarray | list[list[float]], 
    k: int = 10, 
    overfetch: int = 15, 
    batch_size: int = 50
) -> list[list[str]]:
    
    query_embeddings = np.asarray(query_embeddings)
    all_docs = []

    for start in tqdm(range(0, len(query_embeddings), batch_size), desc="Retrieving"):
        batch = query_embeddings[start:start + batch_size]
        results = collection.query(query_embeddings=batch.tolist(), n_results=k + overfetch)

        for docs in results["documents"]: # type: ignore
            seen = set()
            unique_docs = []
            
            for d in docs:
                if d not in seen:
                    seen.add(d)
                    unique_docs.append(d)
                    
                if len(unique_docs) == k:
                    break

            if len(unique_docs) < k:
                print(f"Warning: only {len(unique_docs)}/{k} unique chunks retrieved")

            all_docs.append(unique_docs)

    for i, docs in enumerate(all_docs):
        if len(docs) < k: print(f"Row {i} has {len(docs)} chunks, expected {k}")

    return all_docs

In [11]:
# # Get embeddings for test data
# embedder = SentenceTransformer(    
#     model_config["embedder_model"],
#     model_kwargs={
#         "trust_remote_code": True,
#         "dtype": torch.float16,
#         "device_map": "cuda:0"
#     },
#     processor_kwargs={"padding_side": "left"},
#     prompts={"mcq_query": f"Instruct: {MCQ_EMBED_INSTRUCTION}\nQuery: "},
#     default_prompt_name="mcq_query"
# )

# test_query_embeddings = embedder.encode(
#     test_data["mcq_query"].to_list(), 
#     show_progress_bar=True,
#     batch_size=32
# ).astype(np.float32) # type: ignore

# del embedder
# reset_gpu()

In [12]:
# # Get retrieved chunks for test data
# test_retrieved_chunks = retrieve_top_k(test_query_embeddings, k=10)
# test_data = test_data.with_columns(
#     pl.Series("retrieved_chunks", test_retrieved_chunks)
# )
# test_data.write_parquet(inter_path_config["test_retrieved"])

In [13]:
%%writefile rerank_worker.py
# Define script to parallelize ranking of retrieved documents
import gc, sys, torch, argparse

from tqdm.auto import tqdm

import numpy as np
import polars as pl

from pathlib import Path

from sentence_transformers import CrossEncoder
from transformers import BitsAndBytesConfig

def main() -> None:
    p = argparse.ArgumentParser()
    p.add_argument("--input", required=True)
    p.add_argument("--output", required=True)
    p.add_argument("--model_path", required=True)
    p.add_argument("--instruction", required=True)
    p.add_argument("--batch_size", type=int, default=16)
    p.add_argument("--k", type=int, default=5)
    args = p.parse_args()

    payload = pl.read_parquet(args.input)
    query_texts = payload["query"].to_list()
    retrieved_chunks = payload["retrieved_chunks"].to_list()

    bnb_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=6.0,
        llm_int8_has_fp16_weight=False
    )

    model = CrossEncoder(
        args.model_path,
        trust_remote_code=True,
        model_kwargs={
            "quantization_config": bnb_config,
            "attn_implementation": "sdpa",
            "dtype": torch.float16,
            "device_map": "cuda:0"
        },
        prompts={"rerank": args.instruction},
        default_prompt_name="rerank"
    )

    if model.tokenizer.pad_token is None:
        model.tokenizer.pad_token = model.tokenizer.eos_token
        
    model.model.config.pad_token_id = model.tokenizer.pad_token_id
    model.model.config.use_cache = False
    model.model.eval()

    flat_pairs, boundaries = [], []
    idx = 0

    for q, chunks in zip(query_texts, retrieved_chunks):
        chunks = [c if c.strip() else " " for c in chunks]
        flat_pairs.extend((q, c) for c in chunks)
        boundaries.append((idx, idx + len(chunks)))
        idx += len(chunks)

    flat_scores = model.predict(
        flat_pairs,
        batch_size=args.batch_size,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    del model
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    top_chunks_per_query = []
    for (start, end), chunks in zip(boundaries, retrieved_chunks):
        scores = flat_scores[start:end]
        top_idx = np.argsort(-scores)[:args.k]
        top_chunks_per_query.append([chunks[i] for i in top_idx])

    pl.DataFrame({"top_chunks": top_chunks_per_query}).write_parquet(args.output)

if __name__ == "__main__":
    main()

Writing rerank_worker.py


In [14]:
# Define function to parallelize ranking of retrieved documents [rerank_top_k]
def rerank_top_k(query_texts, retrieved_chunks, tag, k=5, batch_size=16):
    mid = len(query_texts) // 2
    halves = {
        0: (query_texts[:mid], retrieved_chunks[:mid]),
        1: (query_texts[mid:], retrieved_chunks[mid:]),
    }

    input_paths, output_paths = {}, {}
    for gpu_id, (qs, chunks) in halves.items():
        input_paths[gpu_id] = f"{tag}_{gpu_id}_input.parquet"
        output_paths[gpu_id] = f"{tag}_{gpu_id}_output.parquet"

        pl.DataFrame({"query": qs, "retrieved_chunks": chunks}).write_parquet(input_paths[gpu_id])

    procs = []
    for gpu_id in (0, 1):
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu_id)}
        cmd = [
            sys.executable, "rerank_worker.py",
            "--input", input_paths[gpu_id],
            "--output", output_paths[gpu_id],
            "--model_path", model_config["reranker_model"],
            "--instruction", RAG_RERANK_INSTRUCTION,
            "--batch_size", str(batch_size),
            "--k", str(k)
        ]
        procs.append(subprocess.Popen(cmd, env=env))

    for proc in procs:
        if proc.wait() != 0:
            raise RuntimeError("rerank_worker.py failed - check the cell output above for the traceback")

    top0 = pl.read_parquet(output_paths[0])["top_chunks"].to_list()
    top1 = pl.read_parquet(output_paths[1])["top_chunks"].to_list()
    reset_gpu()
    
    return top0 + top1

In [15]:
# # Get reranked chunks from retrieved chunks for test data
# test_retrieved_chunks = pl.read_parquet(inter_path_config["test_retrieved"])["retrieved_chunks"].to_list()
# test_ranked_chunks = rerank_top_k(
#     test_data["mcq_query"].to_list(), 
#     test_retrieved_chunks, 
#     tag="test"
# )
# test_data = test_data.with_columns(pl.Series("ranked_chunks", test_ranked_chunks))
# test_data.write_parquet(inter_path_config["test_ranked"])

In [16]:
# Delete intermediate files from disk
patterns = ["*_input.parquet", "*_output.parquet", "*_retrieved.parquet"]

for pattern in patterns:
    for file_path in inter_path_config["working_dir"].glob(pattern):
        if file_path.is_file():
            file_path.unlink()
            print(f"Successfully deleted: {file_path.name}")

In [17]:
# # Load tokenizer and model for generative_slm on devices
# reset_gpu()

# tokenizer = AutoTokenizer.from_pretrained(
#     model_config["generative_slm"], 
#     trust_remote_code=True,
#     truncation_side="left",
#     padding_side="left"
# )
    
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token

# OPTION_TOKEN_IDS = {
#     opt: tokenizer(f"{opt}", add_special_tokens=False)["input_ids"][-1]
#     for opt in OPTION_COLS
# }

# model = AutoModelForCausalLM.from_pretrained(
#     model_config["generative_slm"],
#     quantization_config=bnb_config,
#     attn_implementation="sdpa",
#     trust_remote_code=True,
#     dtype=torch.float16, 
#     device_map="auto"
# )
    
# model.config.use_cache = False
# model.eval()

In [18]:
# Define function to build prompt for SLM [build_slm_prompt]
def build_slm_prompt(row: dict, chunks: list[str]) -> str:
    context = "\n\n".join(chunks) if chunks else ""
    
    options_block = "\n".join(
        f"{opt}) {row[opt]}" for opt in OPTION_COLS if row[opt] is not None
    )

    user_content = f"Context:\n{context}\n\nQuestion: {row["prompt"]}\nOptions:\n{options_block}\n\nAnswer:"
    
    messages = [
        {"role": "system", "content": GENERATIVE_SLM_INSTRUCTION},
        {"role": "user", "content": user_content}
    ]

    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [19]:
# Define function to rank all 5 options [rank_options], or rank top 3 options [predict_top3]
def rank_options(
    data: pl.DataFrame, 
    top_chunks: list[list[str]], 
    batch_size: int = 4
) -> list[str]:
    
    rows = data.to_dicts()
    predictions = []
    reset_gpu()

    ALL_OPT_IDS = torch.tensor([OPTION_TOKEN_IDS[o] for o in OPTION_COLS], device=model.device)
    
    with torch.inference_mode():
        for start in tqdm(range(0, len(rows), batch_size), desc="Scoring"):
            batch_rows = rows[start:start + batch_size]
            batch_chunks = top_chunks[start:start + batch_size]

            prompts = [build_slm_prompt(r, c) for r, c in zip(batch_rows, batch_chunks)]
            enc = tokenizer(prompts, padding=True, truncation=True, return_tensors="pt").to(model.device)

            logits = model(**enc).logits[:, -1, :]
            batch_opt_logits = logits[:, ALL_OPT_IDS]

            for i, row in enumerate(batch_rows):
                valid_mask = torch.tensor([row[opt] is not None for opt in OPTION_COLS], device=logits.device)
                valid_opts = [opt for opt, keep in zip(OPTION_COLS, valid_mask.tolist()) if keep]

                row_logits = batch_opt_logits[i][valid_mask]
                ranked = [valid_opts[j] for j in torch.argsort(row_logits, descending=True).tolist()]
                predictions.append(" ".join(ranked))

            reset_gpu()

    return predictions

def predict_top3(
    data: pl.DataFrame, 
    top_chunks: list[list[str]], 
    batch_size: int = 4
) -> list[str]:
    
    return [" ".join(r.split()[:3]) for r in rank_options(data, top_chunks, batch_size)]

In [20]:
# # Get ranked options from SLM
# test_data = pl.read_parquet(inter_path_config["test_ranked"])
# test_ranked_chunks = test_data["ranked_chunks"].to_list()
# test_data = test_data.with_columns(pl.Series("top_preds", rank_options(test_data, test_ranked_chunks)))

In [21]:
# Define helper functions for Reciprocal Rank Fusion (RRF) [scores_to_ranks, apply_rrf]
def scores_to_ranks(scores: np.ndarray) -> np.ndarray:
    order = np.argsort(-scores, axis=-1, kind="stable")
    ranks = np.empty_like(order)
    
    row_idx = np.arange(scores.shape[0])[:, None]
    ranks[row_idx, order] = np.arange(1, scores.shape[1] + 1)
    
    return ranks
 

def apply_rrf(
    rank_lists: list[np.ndarray],
    k: float = 60.0,
    weights: list[float] | None = None
) -> np.ndarray:

    if weights is None:
        weights = [1.0] * len(rank_lists)
        
    fused = np.zeros_like(rank_lists[0], dtype=float)
    for ranks, w in zip(rank_lists, weights):
        fused += w / (k + ranks)
        
    return fused

In [22]:
# Define function to load custom trained model for RRF [load_lstm_model]
from tokenizer import BPETokenizer
from dataset import MCQDataset, MCQCollateFn
from model import BiLSTMCrossAttentionClassifier

def load_lstm_model(tokenizer: BPETokenizer, device: str = "cuda") -> list[BiLSTMCrossAttentionClassifier]:
    lstm_model = []
    for i in range(1, 6):
        model = BiLSTMCrossAttentionClassifier(
            vocab_size=tokenizer.get_vocab_size(),
            embedding_dim=256,
            hidden_dim=192,
            dropout=0.4726607662309273
        )
        
        state_dict = torch.load(
            f"{LSTM_MODEL_PATH}/best-cv-models/f_{i}_best.pt",
            map_location="cpu", 
            weights_only=True
        )
        
        model.load_state_dict(state_dict)
        model.eval().to(device)
        lstm_model.append(model)
        
    return lstm_model

In [23]:
# # Get custom model predictions from LSTM model
# lstm_probs = np.zeros((len(test_data), 5))

# lstm_tokenizer = BPETokenizer.load(f"{LSTM_MODEL_PATH}/bpe_tokenizer.json")

# lstm_BATCH_SIZE = 16
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# test_loader = DataLoader(
#     MCQDataset(
#         test_data,
#         lstm_tokenizer,
#         is_test=True
#     ),
#     batch_size=lstm_BATCH_SIZE,
#     shuffle=False,
#     num_workers=4,
#     pin_memory=True,
#     collate_fn=MCQCollateFn(lstm_tokenizer)
# )

# lstm_models = load_lstm_model(tokenizer=lstm_tokenizer, device=DEVICE)

# for model in lstm_models:
#     test_start_idx = 0
#     all_test_probs = torch.zeros((len(test_data), 5), device=DEVICE)

#     with torch.no_grad():
#         for batch in test_loader:
#             prompt_ids = batch["prompt_ids"].to(DEVICE)
#             prompt_mask = batch["prompt_mask"].to(DEVICE)
#             opt_ids = batch["opt_ids"].to(DEVICE)
#             opt_mask = batch["opt_mask"].to(DEVICE)
                
#             test_logits = model(prompt_ids, prompt_mask, opt_ids, opt_mask)
#             test_probs = F.softmax(test_logits, dim=1)

#             batch_size = test_probs.size(0)
#             all_test_probs[test_start_idx : test_start_idx + batch_size] = test_probs
#             test_start_idx += batch_size

#     all_test_probs = all_test_probs.cpu().numpy()
#     lstm_probs += all_test_probs / len(lstm_models)

#     del model
#     reset_gpu()

# lstm_ranks = scores_to_ranks(lstm_probs)

In [24]:
# # Directly get lstm predictions
# top3_idx = np.argsort(-lstm_probs, axis=1)[:, :3]
# pred_strings = [" ".join(OPTION_COLS[i] for i in row) for row in top3_idx]

In [25]:
# # Parse RAG predictions for RRF
# rag_ranks = np.zeros((len(test_data), 5))

# for i, pred_str in enumerate(test_data["top_preds"].to_list()):
#     preds = pred_str.split()
    
#     for rank, opt in enumerate(preds):
#         if opt in OPTION_COLS:
#             opt_idx = OPTION_COLS.index(opt)
#             rag_ranks[i, opt_idx] = rank + 1

In [26]:
# # Fuse ranks from lstm and rag pipeline
# fused_scores = apply_rrf([rag_ranks, lstm_ranks], k=60.0)

# top3_idx = np.argsort(-fused_scores, axis=1)[:, :3]
# pred_strings = [" ".join(OPTION_COLS[i] for i in row) for row in top3_idx]

In [27]:
# Get predictions from finetuned `zerank-2-reranker` for submission
test_pairs = []

for row in test_data.iter_rows(named=True):
    for option in OPTION_COLS:
        option_text = row[option] if row[option] is not None else " "
        test_pairs.append((row["prompt"], str(option_text)))

reranker = CrossEncoder(
    BASE_RERANKER_PATH,
    num_labels=1,
    model_kwargs={
        "quantization_config": bnb_config,
        "device_map": "cuda:0"
    },
    prompts={"rerank": MCQ_RERANK_INSTRUCTION},
    default_prompt_name="rerank"
)

peft_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

reranker.add_adapter(peft_config)

n_samples = len(test_data)
rank_probs = np.zeros((n_samples, 5))

for fold in range(5):
    state_dict_path = f"{MCQ_ADAPTER_PATH}/qlora-mcq-reranker-best-fold-{fold}.pt"
    adapter_state_dict = torch.load(
        state_dict_path, 
        map_location="cpu",
        weights_only=True
    )
    set_peft_model_state_dict(reranker.model, adapter_state_dict)
    
    fold_logits = reranker.predict(
        test_pairs,
        batch_size=4,
        show_progress_bar=True
    )

    fold_logits_reshaped = torch.tensor(fold_logits).view(n_samples, 5)
    fold_probs = fold_logits_reshaped.sigmoid()
    fold_probs = fold_probs.numpy()
    
    rank_probs += fold_probs / 5

top3_idx = np.argsort(-rank_probs, axis=1)[:, :3]
pred_strings = [" ".join(OPTION_COLS[idx] for idx in row) for row in top3_idx]

Default prompt name is set to 'rerank'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

In [28]:
# Create and save the final submission file
submission = test_data.select(["id"]).with_columns(
    pl.Series("Prediction", pred_strings)
).rename({"id": "ID"})

submission.write_csv(inter_path_config["submission"])